# Dust Coagulation Tutorial 2 - Emulation

In this tutorial, we will use the simulation data that we have generated in the first tutorial to train a simple neural network emulator for the dust coagulation simulation.

For the purposes of this, we will be using the [pytorch](https://pytorch.org/get-started/locally/) package to implement the neural network approach. You can reuse the python environment that we have created in the first tutorial, but you will still have to install pytorch, as the version depends on whether you have a GPU available or not. Follow the above link to the pytorch website and use the provided terminal commands to install the appropriate version. That is for your laptop, you will likely use the CPU-only version, whereas for the compute-server you can use one of the GPU-enabled versions. For the latter, check which version of CUDA is run on the compute servers, using the `nvidia-smi` command from the shell.

If you are not familiar with pytorch at all, please have a look at the following introduction for the basic usage of the package:
https://docs.pytorch.org/tutorials/beginner/basics/intro.html

In [ ]:
import os
import numpy as np
import pandas
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from time import time, gmtime

## 1. Setup the training data 

To train the neural network, we now have to set up the dedicated training, validation and test data sets. In pytorch these datasets will be handled by `torch.utils.data.DataLoader` objects, which can automatically take care of the batching of the data that we use during the stochastic gradient descent.

These dataloaders take a `torch.utils.data.TensorDataset` and the batch size as an input. The TensorDataset takes two `torch.tensors` as a input, one defining the input parameters x and one that defines the output parameters or targets of the network y.

For the input parameters use $\left[\log(\rho_\mathrm{gas}), \log(D/G), \log(\alpha_\mathrm{turb}), \rho_\mathrm{grain}, \log(\mathrm{dt_i}), \log{\rho}_\mathrm{dust}^0(t=0), \ldots, \log{\rho}_\mathrm{dust}^{N_\mathrm{bins}}(t=0)\right]$.

For the target parameters use $\left[\log{\rho}_{\mathrm{dust}, i}^0(\mathrm{dt_i}), \ldots, \log{\rho}_{\mathrm{dust}, i}^{N_\mathrm{bins}}(\mathrm{dt_i})\right]$.


In the following setup three dataloaders, one for training, one for testing and one for validation. The data should be split randomly into these datasets (use a fixed seed for reproducability). Account 70% of the generated data as training, 20% for testing and 10% for validation.

For visualisation purposes, also hold out a few (10-20) initial condition configurations, before you do the random splitting, just so we have a few full consecutive full simulations to look at.



*Bonus objective:*

It is common practise in ML training procedures to re-scale the data before it is fed into the neural network. A common rescaling procedure is **standardisation**. Here, we rescale all input features and target parameters as follows

$\tilde{x} = (x - \mu_x) / \sigma_x$, 

where $\mu_x$, $\sigma_x$ are the mean and standard deviation of feature/target $x$ over the training data. Using this procedure the distribution of each feature/target in the training data will have zero mean and unit standard deviation. Rescaling procedures like this serve as a means to e.g. compensate for differences in magnitude between the target parameters, which could lead to certain target parameters overly driving the gradient of the loss just by virtue of their magnitude. The coefficients for these rescaling procedures are always derived on the training data and then applied in the same way to all other data sets that are processed (i.e. validation, test, prediction).

Note that at prediction time, if you are using such rescaling procedures, the network output has to be transformed back to the original target parameter space. As the rescaling procedures are typically chosen to be fairly simple linear transformations, these re-re-scaling steps are trivial. For standardisation, we have for example

$ x = \tilde{x} * \sigma_x + \mu_x$.

As we are operating almost exclusively on logarithmic quantities in this tutorial, however, an additional rescaling procedure is not strictly required for the following, as the logarithmic scaling is a rescaling procedure in itself already.


In [ ]:
# TODO: Load training data and split into training, validation and test sets for NN training


## 2. Setup the neural network architecture

Next we will setup up the neural network architecture. In this tutorial, we will be building simple fully connected neural networks with a few hidden layers. For this purpose, you have a class template provided below. Fill in the missing methods.

This includes:

1. The initialisation of the network. Set it up such that you can provide a list of sizes to use for the hidden layers. In addition, have the option to switch between ReLU and and SiLU activation functions. Lastly, initialise the model on the specified device (i.e. on CPU or GPU). Hint: You will need `torch.nn.Linear`, `torch.nn.ReLU`, `torch.nn.SiLU`, `torch.nn.Sequential`. Also make sure that you do not apply an activation function to the output layer.
2. Prepare the training procedure in `setup_training`. Initialise the gradient descent optimiser, learning rate scheduler and loss function to use. Use Adam for the optimiser, a StepLR for the learning rate scheduling and an MSE loss function. Also initialise the network weights by sampling from a normal distribution and allow for an adjustment of the amplitude with the `init_scale` parameter.
3. Implement the training procedure in `run_training`. For a specified number of epochs, loop over all batches of the training data, perform the gradient descent on the training data and then compute the validation loss on the validation data. For every epoch keep track of the mean training and validation losses. Hint: `torch.tensor` keep track of all operations that are being performed on them. To backpropagate the loss, you therefore have to only call the `.backward()` method on the output of the loss function. When you are computing the loss on the validation data, however, you do not want to accumulat any gradients at all, in order to make sure that your validation data does not somehow affect the weight updates. This can be done using e.g. `with torch.no_grad():`, when you let the network process the validation inputs.
4. Implement the method to test the method on a held-out test set given the test dataloader in `run_test`. Return two arrays here, one with the ground truth values for each test example and one with the network predictions.
5. Implement the saving and loading of the network weights in the `save` and `load` methods.

For your convenience, the class template includes a custom plotting method to visualise the simple network architecture that you are going to built. You can test whether your initialisation works correctly, if the `plot_architecture` works. 


In [ ]:
from typing import List, Tuple
from matplotlib.lines import Line2D

# Simple network class
class FullyConnectedNetwork(torch.nn.Module):
    def __init__(self, dim_in: int, dim_out: int, layer_sizes: List[int], device: str, activation: str='ReLU'):
        super().__init__()
        nlayers = len(layer_sizes)
        self.dim_in = dim_in
        self.dim_out = dim_out
        self.device = device
        self.layer_sizes = layer_sizes
        self.activation = activation

        ## TODO: Add initialisation of the network here
        # self.model = 
        raise NotImplementedError()

    def forward(self, x: torch.tensor) -> torch.tensor:
        return self.model(x)


    def setup_training(self, lr_init: float, l2_weight_reg: float, gamma: float, adam_betas: Tuple[float], init_scale: float=0.03):
        raise NotImplementedError()
        # TODO: Add initialisation of methods for the training procedure here
        # self.optimiser = 
        # self.scheduler =
        # self.loss_func = 
    
    def run_training(self, n_epochs: int, train_loader: torch.utils.data.DataLoader, val_loader: torch.utils.data.DataLoader, verbose: bool=True):
        
        self.loss_curve = np.zeros((n_epochs, 3))
        self.loss_curve[:, 0] = np.arange(n_epochs)
        
        t_start = time()

        if verbose:
            print("######### Starting Training #########") 
            print('│{:^9}│{:^15}│{:^11}|{:^16}|'.format('Epoch', 'Time elapsed', 'MSE Train', 'MSE Validation'))
            print('|' + '-' * 9 + '|' + '-' * 15 + '|' + 11 * '-' + '|' + 16 * '-' + '|')
        
        for i_epoch in range(n_epochs):
            ## TODO: Implement the training procedure here
            raise NotImplementedError()
            
            t_elapsed = time() - t_start
            t_gm = gmtime(t_elapsed)

            if verbose:
                print('│{:^9}│{:^15}│{:^11}|{:^16}|'.format(f'{i_epoch:02d}', 
                                                            f'{t_gm.tm_hour:02d}h{t_gm.tm_min:02d}m{t_gm.tm_sec:02d}s',
                                                            f'{self.loss_curve[i_epoch, 1]:.4f}', 
                                                            f'{self.loss_curve[i_epoch, 2]:.4f}'))
        if verbose:  
            print('|' + '-' * 9 + '|' + '-' * 15 + '|' + 11 * '-' + '|' + 16 * '-' + '|')
            t_end = time()
            print(f'Training took {(t_end - t_start) / 60:.2f}min')

        self.print_loss_curve()

    def run_test(self, test_loader: torch.utils.data.DataLoader) -> Tuple[np.array]:
        test_pred = []
        test_gt = []

        # TODO: Implement application to a test data set.
        raise NotImplementedError()
        
        return test_gt, test_pred
    
    def print_loss_curve(self):
        
        f, ax = plt.subplots(1, 2, figsize=(5 * 2, 5), tight_layout=True)
        ax[0].plot(self.loss_curve[:, 0], self.loss_curve[:, 1], c="black", label="Training", lw=2)
        ax[0].plot(self.loss_curve[:, 0], self.loss_curve[:, 2], c="orange", label="Validation", lw=2)
        ax[0].set_ylabel("MSE Loss", fontsize=18)
        ax[0].set_xlabel("Training Epoch", fontsize=18)
        ax[0].legend(loc="best", fontsize=14)
        ax[0].tick_params(width=2, labelsize=15)
        plt.setp(ax[0].spines.values(), linewidth=2)
        
        ax[1].plot(self.loss_curve[:, 0], np.log10(self.loss_curve[:, 1]), c="black", label="Training", lw=2)
        ax[1].plot(self.loss_curve[:, 0], np.log10(self.loss_curve[:, 2]), c="orange", label="Validation", lw=2)
        ax[1].set_ylabel("Log MSE Loss", fontsize=18)
        ax[1].set_xlabel("Training Epoch", fontsize=18)
        ax[1].legend(loc="best", fontsize=14)
        ax[1].tick_params(width=2, labelsize=15)
        plt.setp(ax[1].spines.values(), linewidth=2)

    def save(self, path: str):
        # TODO: Implement saving of the network weights
        raise NotImplementedError()

    def load(self, path: str):
        # TODO: Implement loading of the network weights
        raise NotImplementedError()
    
    def plot_architecture(self):

        nlayers = len(self.layer_sizes) + 2

        sizes = [self.dim_in] + self.layer_sizes + [self.dim_out]

        max_size = max(sizes)
        min_size = min(sizes)

        sizes_scaled = np.array(sizes) / max_size
        sizes_scaled = sizes_scaled * 1.8
        
        xpos = np.linspace(0, 10, nlayers, endpoint=True)
        ypos = 1 - sizes_scaled / 2

        width = (xpos[1] - xpos[0]) / 2

        
        f, ax = plt.subplots(1, 1, figsize=(10, 5), tight_layout=True)

        ax.set_axis_off()
        ax.set_xlim(-1, 11)
        ax.set_ylim(-0.25, 2)

        rect_in = plt.Rectangle([xpos[0] - width / 2, ypos[0]], width, sizes_scaled[0], facecolor="orange", edgecolor="black")
        ax.add_patch(rect_in)
        ax.text(xpos[0], ypos[0] + sizes_scaled[0] / 2, self.dim_in, ha='center', va='center', c="black", fontsize=12)
        ax.text(xpos[0], ypos[0] + sizes_scaled[0], "Input", ha='center', va='bottom', c="black", fontsize=12)

        rect_out = plt.Rectangle([xpos[-1] - width / 2, ypos[-1]], width, sizes_scaled[-1], facecolor="lightblue", edgecolor="black")
        ax.add_patch(rect_out)
        ax.text(xpos[-1], ypos[-1] + sizes_scaled[-1] / 2, self.dim_out, ha='center', va='center', c="black", fontsize=12)
        ax.text(xpos[-1], ypos[-1] + sizes_scaled[-1], "Output", ha='center', va='bottom', c="black", fontsize=12)

        lk = 0
        
        for n in np.arange(1, nlayers - 1, 1):

            rect_n = plt.Rectangle([xpos[n] - width / 2, ypos[n]], width * 0.75, sizes_scaled[n], facecolor="lightgrey", edgecolor="black")
            rect_act_n = plt.Rectangle([xpos[n] + width / 4, ypos[n]], width / 4, sizes_scaled[n], facecolor="salmon", edgecolor="black")
    
            ax.add_patch(rect_n)
            ax.add_patch(rect_act_n)

            ax.text(xpos[n] - 0.125 * width, ypos[n] + sizes_scaled[n] / 2, self.layer_sizes[lk], fontsize=12,
                    ha='center', va='center', rotation=90)
            ax.plot([xpos[n-1] + width / 2, xpos[n] - width / 2], 
                    [ypos[n-1] + sizes_scaled[n-1], ypos[n] + sizes_scaled[n]],
                    c="black", lw=1)
            ax.plot([xpos[n-1] + width / 2, xpos[n] - width / 2],  [ypos[n-1], ypos[n]], c="black", lw=1)
            
            lk += 1

        ax.plot([xpos[-2] + width / 2, xpos[-1] - width / 2], 
                [ypos[-2] + sizes_scaled[-2], ypos[-1] + sizes_scaled[-1]],
                c="black", lw=1)
        ax.plot([xpos[-2] + width / 2, xpos[-1] - width / 2],  [ypos[-2], ypos[-1]], c="black", lw=1)

        custom_lines = [Line2D([0], [0], color='lightgrey', lw=18), Line2D([0], [0], color='salmon', lw=18)]
    
        ax.legend(custom_lines, ['Linear', self.activation], loc='lower center', fontsize=18, ncol=2)

## 3. Train the neural network

With the wrapper class completed, initialise a network configuration now, plot the architecture, and then setup and run the training procedure. For this first test, use a fully connected network with **ReLU() activation** and **three hidden layers** of size **1024**. Also make sure to **save the network weights** after training. 
For the training hyperparameters you can start with the following setup:

In [ ]:
# Hyperparameters for training
lr_init = 0.001             # Initial Learning rate
l2_weight_reg = 1e-5        # Strength of L2 weight regularisation
init_scale = 0.03           # Amplitude for Gaussian weight initialisation
adam_betas = (0.9, 0.999)   # Betas of ADAM optimiser
gamma = 0.9                 # Weight decay factor  
n_epochs=60                 # Number of training epochs

In [ ]:
# TODO: Initialise and visualise the network architecture

In [ ]:
# TODO: Setup and run the training procedure

## 4. Test the neural network

With the network trained, we can now investigate the performance on the held-out test data.

### 4.1 Time the network prediction

First, let's have a look at the execution time of the trained emulator. Use a subset of the held-out test data or visualisation data set of 100 examples and use `%timeit` again to get some statistics on the prediction time with the neural network emulator. If you have the option, you may also test the execution time between running the network on CPU and GPU. 

In [ ]:
# TODO: Implement timing of the trained emulator for a subset of 100 test examples.

### 4.2 Visualise some example predictions

Let's now look at how well the network manages to approximate the coagulation simulation. Use the smaller held-out visualisation data set and run the network prediction on it.
Then compare the network predictions of the dust grain size distribution to the ground truth grain size distribution for different time steps for a fixed set of initial conditions by plotting the dust grain size distribution.

In [ ]:
# TODO: Run network on visualisation data set and plot the emulation results in comparison to the actual simulation results

### 4.3 Compute performance over the test set

Next up, we are going to get a quantitative measure of the performance of the neural network emulator on the whole held-out test data. For this purpose a helper-function has been provided below which prepares to plots of the test performance given the ground truth and predicted dust grain distributions. In particular, this function will create a 2D histogram plot of the 1-to-1 correlation between prediction and ground truth for each bin of the grain size distribution. In addition, it will also verify, whether the network adheres to the mass-conservation constraint, by comparing the predicted sums of the grain densities to the expected value, again in a 2D histogram. 

In [ ]:
from matplotlib import colors

def plot_test_results(test_gt: np.array, test_pred: np.array, network_suffix: str):

    PARAM_RANGES = np.max(train_y.numpy(), axis=0) - np.min(train_y.numpy(), axis=0)
    NCOL = 10
    NROW = 10
    NBINS = 100
    
    # Calculate mean squared error and normalised RMSE of the predictions
    rmse = np.sqrt(np.nanmean((test_pred - test_gt)**2, axis=0))
    nrmse = rmse / PARAM_RANGES
    
    # Make plot
    ydim = train_y.shape[1]
    panel_width = 4
    panel_height = 4
    
    hratios = [0.15] + [0.1, 1, 0.3] * NROW
    hratios[-1] = 0.2
    wratios = [0.3, 1] * NCOL + [0.125]
    
    gs = plt.GridSpec(nrows=len(hratios),
                      ncols=len(wratios),
                      height_ratios=hratios,
                      width_ratios=wratios,
                      hspace=0.0, wspace=0,
                      left=0.00, right=1,
                      bottom=0.0, top=1)
    
    f = plt.figure(figsize=(panel_width * sum(wratios), panel_height * sum(hratios)))
    
    headspace = f.add_subplot(gs[0, :])
    headspace.set_axis_off()
    
    row_k = 0
    col_k = 0
    
    for k in range(NROW * NCOL):
    
        if k < ydim:
            # add panel
            ax_k = f.add_subplot(gs[2 + row_k * 3, 1 + col_k * 2])
            ax_cbar_k = f.add_subplot(gs[1 + row_k * 3, 1 + col_k * 2])
    
            # Compute axis limits
            x_min, x_max = np.min(test_gt[:, k]), np.max(test_gt[:, k])
            lim_x_min = x_min - 0.04 * (x_max - x_min)
            lim_x_max = x_max + 0.04 * (x_max - x_min)
    
            ax_k.plot(np.linspace(lim_x_min, lim_x_max, 100),
                      np.linspace(lim_x_min, lim_x_max, 100),
                      lw=1.5, ls='dashed', c='black', zorder=1.5)
    
            # 2d density plot with log colour scaling
            pk = ax_k.hist2d(test_gt[:, k], test_pred[:, k],
                             bins=(NBINS, NBINS), cmap='inferno_r',
                             range=[[np.min(test_gt[:, k]), np.max(test_gt[:, k])],
                                    [np.nanmin(test_pred[:, k]), np.nanmax(test_pred[:, k])]],
                             zorder=2)
            pk[3].norm = colors.LogNorm()
            cbk = f.colorbar(pk[3], cax=ax_cbar_k, orientation='horizontal', ticklocation='top')
            cbk.set_label("Counts", labelpad=-47.5, fontsize=18, color="white")
            cbk.ax.tick_params(labelsize=18, width=2, length=6)
            cbk.ax.tick_params(which='minor', width=1.5, length=4)
            cbk.outline.set_linewidth(2)
    
            ax_k.set_xlabel('$\\log(\\rho_{%i})^\\mathrm{GT}$' % k, fontsize=20)
            ax_k.set_ylabel('$\\log(\\rho_{%i})^\\mathrm{pred}$' % k, fontsize=20)
            ax_k.set_xlim(lim_x_min, lim_x_max)
            ax_k.set_ylim(lim_x_min, lim_x_max)
            ax_k.tick_params(width=2, labelsize=16)
    
            txt_k = r'$\mathrm{RMSE} = %.3f$' % (rmse[k])
            txt_k += '\n' + r'$\mathrm{NRMSE} = %.3f$' % (nrmse[k])
            ax_k.text(0.97, 0.03,
                      txt_k,
                      verticalalignment='bottom',
                      horizontalalignment='right',
                      bbox=dict(facecolor='white', alpha=0.5),
                      transform=ax_k.transAxes,
                      fontsize=16)
            plt.setp(ax_k.spines.values(), linewidth=2)
    
            # Update panel index
            col_k += 1
            if col_k == NCOL:
                row_k += 1
                col_k = 0
    
    f.savefig(f"{PATH_OUTPUT}/Hist2D_Pred_vs_GT{network_suffix}.pdf")

    # Plot mass conservation
    log_mass_gt = np.log10(np.sum(10**test_gt, axis=1))
    log_mass_pred = np.log10(np.sum(10**test_pred, axis=1))
    
    rmse = np.sqrt(np.nanmean((log_mass_pred - log_mass_gt)**2, axis=0))
    nrmse = rmse / (log_mass_gt.max() - log_mass_gt.min())
    
    NBINS = 100
    panel_width = 4
    panel_height = 4
    
    hratios = [0.15, 0.1, 1, 0.2]
    wratios = [0.3, 1, 0.125]
    
    gs = plt.GridSpec(nrows=len(hratios),
                      ncols=len(wratios),
                      height_ratios=hratios,
                      width_ratios=wratios,
                      hspace=0.0, wspace=0,
                      left=0.00, right=1,
                      bottom=0.0, top=1)
    
    f = plt.figure(figsize=(panel_width * sum(wratios), panel_height * sum(hratios)))
    
    headspace = f.add_subplot(gs[0, :])
    headspace.set_axis_off()
    
    row_k = 0
    col_k = 0
    
    ax_k = f.add_subplot(gs[2 + row_k * 3, 1 + col_k * 2])
    ax_cbar_k = f.add_subplot(gs[1 + row_k * 3, 1 + col_k * 2])
    
    # Compute axis limits
    x_min, x_max = np.min(log_mass_gt), np.max(log_mass_gt)
    lim_x_min = x_min - 0.04 * (x_max - x_min)
    lim_x_max = x_max + 0.04 * (x_max - x_min)
    
    ax_k.plot(np.linspace(lim_x_min, lim_x_max, 100), np.linspace(lim_x_min, lim_x_max, 100), lw=1.5, ls='dashed', c='black', zorder=1.5)
    
    # 2d density plot with log colour scaling
    pk = ax_k.hist2d(log_mass_gt, log_mass_pred, bins=(NBINS, NBINS), cmap='inferno_r',
                     range=[[np.min(log_mass_gt), np.max(log_mass_gt)],
                            [np.nanmin(log_mass_pred), np.nanmax(log_mass_pred)]],
                     zorder=2)
    pk[3].norm = colors.LogNorm()
    cbk = f.colorbar(pk[3], cax=ax_cbar_k, orientation='horizontal', ticklocation='top')
    cbk.set_label("Counts", labelpad=-47.5, fontsize=18, color="white")
    cbk.ax.tick_params(labelsize=18, width=2, length=6)
    cbk.ax.tick_params(which='minor', width=1.5, length=4)
    cbk.outline.set_linewidth(2)
    
    ax_k.set_xlabel('$\\log\\left(\\Sigma_i \\rho_i\\right)^\\mathrm{GT}$', fontsize=20)
    ax_k.set_ylabel('$\\log\\left(\\Sigma_i \\rho_i\\right)^\\mathrm{pred}$', fontsize=20)
    ax_k.set_xlim(lim_x_min, lim_x_max)
    ax_k.set_ylim(lim_x_min, lim_x_max)
    ax_k.tick_params(width=2, labelsize=16)
    
    txt_k = r'$\mathrm{RMSE} = %.4f$' % (rmse)
    txt_k += '\n' + r'$\mathrm{NRMSE} = %.4f$' % (nrmse)
    ax_k.text(0.97, 0.03,
              txt_k,
              verticalalignment='bottom',
              horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5),
              transform=ax_k.transAxes,
              fontsize=16)
    
    plt.setp(ax_k.spines.values(), linewidth=2)

    f.savefig(f"{PATH_OUTPUT}/Hist2D_MassConservation{network_suffix}.pdf")

In [ ]:
# TODO: Run the network on the held out test data and use the provided helper function to plot the network performance

### 4.4 Investigate the cumulative error

The main goal of emulator development is to provide a more efficient tool for computing certain physics in a given simulation, so that the simulation may become more complex without blowing up computation costs to an unfeasible degree. In this use case, the trained emulator will be executed on its own outputs. We must therefore check, whether the emulator is robust with respect to a consecutive application to its own outputs.

Use the smaller visualisation data set and setup a prediction run, where the emulator is consecutively applied to its own output for a fixed set of initial conditions. That is rather than directly predicting the grain size distribution for e.g. time step $\mathrm{dt}_i$, as the network is currently trained to do, let it predict the grain size distribution for all intermediate time steps, using the output grain size distribution of the previous time step as the input for the next. Make sure to adjust $\mathrm{dt}$ in this case as well to represent the difference between two consecutive time steps rather than the difference to $t=0$.

For every incremental timestep you can then compare the predicted distribution to the ground truth distribution and compute for example a mean squared error to quantify the cumulative error that the emulator produces. 

If you want to be particularly quantitative, average this over the all initial conditions held-out in the visualisation dataset. Make sure to normalise $\mathrm{dt}$ to the free fall time in this case for comparability.

In [ ]:
# TODO: Apply the network recursively to its own output and quantify the cumulative error of the emulator.

## 5. Experiment with architecture

Now that we have established a baseline for the emulator performance, play around with the network architecture and see if you can find a setup that performs better than our reference model. 

To start off, you can simply vary for example the number of layers, layer size or activation function of the network architecture.

In [ ]:
# TODO: Experiment with neural network architecture

### 5.1 Add physics informed loss

As discussed in the lecture, we may also want to include some prior physical knowledge that we have about the coagulation simulation in our design of the neural network emulator. In this case a very straight forward thing to include, other than adjusting the priors of the generated training data, would be to add a physics-informed loss term to the network training. 
For our setup here, there is a very simple physical constraint that we can formulate as an additional loss term, namely the mass conservation constraint. That is, the sum of the dust grain size distribution has to be the same before and after the time step:

$\sum_{i=1}^{N_\mathrm{bins}} \rho_\mathrm{dust}^{i} = \mathrm{const}$.

Try to implement this constraint as a loss term in the training procedure.

In [ ]:
# TODO: Add physics-informed loss term to the training procedure

### 5.2 Revise training objective with the alternative training data set

As you will likely have seen in the cumulative error testing, the error of the emulator appears to compund quickly. A potential reason for this could be the way that we trained the network, i.e. predicting the dust grain size distribution at various times $\mathrm{dt}$ given the initial conditions and the dust grain size distribution at time $t=0$. 
Alternatively, we could train the network in an incremental way, that is given the initial conditions, grain size distribution at time $\mathrm{dt}_{i-1}$ and the time difference $\Delta t=\mathrm{dt}_i - \mathrm{dt}_{i-1}$, predict the dust grain size distribution at time $\mathrm{dt}_i$. This is why we have created the alternative training data setup for in the previous tutorial.

Retrain the network for this alternative setup as we have done above and investigate the performance.

In [ ]:
# TODO: Train a network on the alternative training data set and report the performance.

## 6. Hyperparameter optimisation

Once you have converged on an architecture/setup that looks promising, the next step is to optimise the hyperparameters of your architecture/training scheme to further improve the performance of your model.

For this purpose, you can of course write your own hyperparameter search routine, but there are also frameworks available that can handle the search for you. In the python environment that was provided for the previous tutorial, one of these packages has already been included, namely [optuna](https://optuna.org/).

Have a look at the following example for a basic example of using optuna:
https://github.com/optuna/optuna-examples/blob/main/pytorch/pytorch_simple.py

Try to rewrite the setup of our network to fit within the optuna framework and then try to run a hyperparameter search. Some hyperparameters to vary are batch_size, initial learning rate, number of layers, layer size, weight decay, strength of the L2 weight regularisation or the meta epoch in the StepLR learning rate scheduling.